[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/Intro_AdFilt_KF.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Adaptive Filtering: the Kalman Filter

The [APA workshop](./Intro_AdFilt_APA.ipynb) adapted a filter by projecting onto recent data. The Kalman filter goes further: it carries a full **statistical belief** about a hidden state — mean *and* uncertainty — and updates that belief optimally with every measurement. It runs in every GPS receiver, drone autopilot, and tracking radar you've ever used.

## 0. Introduction

Setup: a hidden state $\mathbf{x}$ evolves in time; we only see noisy measurements $\mathbf{z}$.

$$\mathbf{x}_k = F\,\mathbf{x}_{k-1} + \mathbf{w}_k \qquad \mathbf{w}_k \sim \mathcal{N}(0, Q)$$
$$\mathbf{z}_k = H\,\mathbf{x}_k + \mathbf{v}_k \qquad\;\; \mathbf{v}_k \sim \mathcal{N}(0, R)$$

$F$: how the state moves. $H$: what we can see of it. $Q$: how much the motion surprises us. $R$: how noisy the sensor is. The filter alternates two beats forever: **predict** (push the belief through $F$, uncertainty grows) and **update** (blend in $\mathbf{z}_k$, uncertainty shrinks).

## 1. Pre-requisites

- [Adaptive Filtering: APA](./Intro_AdFilt_APA.ipynb) — the predict/err/correct loop.
- Basic probability (mean, variance, Gaussian) — see [Random Variables](../Intro_Math/Analysis/README.md#5-random-variables-draft--pending-review) for the rigorous track.
- Matrix multiplication.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)

---
### 🕐 Session 1 of 2 — *State-Space Models & the Kalman Equations* (~35 min)
**Goal:** understand the predict/update cycle and why the Kalman gain is the optimal blend.
**Builds on:** [APA workshop](./Intro_AdFilt_APA.ipynb). &nbsp; **Feeds into:** Session 2 (a real tracking problem).

---

## 2. Theory: Predict, Then Update

💡 **Intuition.** You're estimating where a friend is walking. Your **prediction** ("they were heading north at 1 m/s") and a **measurement** ("I glimpsed them over there") disagree. How to combine? Trust each *in proportion to its certainty*. The Kalman gain $K$ is exactly that trust dial, recomputed every step from the two uncertainties: $K \to 1$ when the sensor is sharp and the model vague; $K \to 0$ when the model is sharp and the sensor noisy. Everything else is bookkeeping for "how uncertain am I now?"

### 2.1. The Five Equations

**Predict** (belief moves with the model, uncertainty inflates):
$$\hat{\mathbf{x}}_k^- = F\hat{\mathbf{x}}_{k-1} \qquad P_k^- = F P_{k-1} F^T + Q$$

**Update** (blend in the measurement):
$$K_k = P_k^- H^T (H P_k^- H^T + R)^{-1}$$
$$\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_k^- + K_k(\mathbf{z}_k - H\hat{\mathbf{x}}_k^-) \qquad P_k = (I - K_k H) P_k^-$$

The term $\mathbf{z}_k - H\hat{\mathbf{x}}_k^-$ is the **innovation** — the part of the measurement the model failed to predict. Notice the shape of the state update: *new estimate = prediction + gain × error*. It's the same heartbeat as LMS/NLMS/APA — the Kalman filter is the member of the family whose gain is **derived** rather than tuned.

*(Full derivation — minimizing $\mathrm{tr}(P_k)$ over all linear unbiased estimators — is left for the session discussion; see any of Kay, Haykin, or Simon for the classical proof.)*

### 2.2. Scalar Sanity Check

A constant hidden value measured in noise. The filter should converge to a running average whose uncertainty shrinks like $1/k$.

In [ ]:

# YOUR CODE HERE


**What just happened.** Final estimate **4.986 ± 0.129** against a truth of 5.0 — and that uncertainty is not approximately right, it is *exactly* right. For a static state with $Q = 0$ and $R = 1$, the posterior variance after $k$ measurements is $P_k = R/k$, so at $k = 60$ the predicted spread is $1/\sqrt{60} = 0.1291$. The filter printed 0.129.

That makes this an oracle check rather than a demonstration. We know the closed-form answer, and the recursion reproduces it — which tells us the implementation is right and, more interestingly, that the Kalman filter in this degenerate case *is* the running average. Trace the gain: $K = P/(P+1)$ starts near 1 while $P = 100$ (an ignorant prior, so the first measurement is taken almost at face value) and decays like $1/k$ as evidence accumulates. Each new measurement moves the estimate less than the last, which is exactly what averaging does.

**The $1/\sqrt{k}$ is the familiar one.** Uncertainty shrinking as the square root of the sample count is the same central-limit scaling that governs every estimator in this curriculum — the confidence bands in [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb), the ACF noise floor, Monte Carlo error. To halve your uncertainty you need four times the data, and no amount of filtering cleverness changes that.

**Read the shaded band as a claim, not decoration.** The ±2σ region is the filter's own assertion that the truth lies inside it about 95% of the time, and the dashed truth line does stay within it. A band that the truth escapes far too often means the filter is overconfident — usually $Q$ set too small — and one it never approaches means the opposite. This is the calibration check that Session 2's tuning discussion is really about, and here it passes.

**What makes this case easy.** The state never moves, so there is no model error to trade against measurement error, and the answer converges monotonically. Session 2 gives the state genuine dynamics and only partial observability — position measured, velocity hidden — at which point the balance between $Q$ and $R$ starts to matter.

---
### 🕐 Session 2 of 2 — *Tracking in Practice* (~40 min)
**Goal:** build a constant-velocity tracker, tune Q and R, and see what mistuning does.
**Builds on:** Session 1.

---

## 3. Application: Tracking a Moving Target

### 3.1. The Model

State = position and velocity, $\mathbf{x} = [p, v]^T$; we measure only position. With time step $\Delta t$:

$$F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}, \quad H = [1 \;\; 0]$$

"Constant velocity" is a *statistical* statement: velocity persists, plus random acceleration kicks captured by $Q$.

In [ ]:

# YOUR CODE HERE


In [ ]:
        # predict
        # update

# YOUR CODE HERE


The filter also estimates **velocity — a quantity we never measured**. That's the quiet superpower of state-space models: observable combinations of hidden states get estimated for free.

In [ ]:

# YOUR CODE HERE


**What just happened.** The estimated velocity tracks the true velocity — and velocity was **never measured**. Look at $H = [1\;\;0]$: it selects position and discards velocity entirely, so no measurement in this entire run contains direct information about how fast the target is moving.

**How the filter recovers it.** The state-transition matrix $F$ couples the two: today's velocity becomes tomorrow's position. So an error in the velocity estimate does not stay hidden — it produces a *systematic drift* in the position innovations, a consistent tendency to predict too far ahead or too far behind. The covariance matrix $P$ carries the position–velocity correlation that converts that pattern into a velocity correction. The filter infers what it cannot see from how what it *can* see evolves over time.

This is the quiet superpower of state-space models, and it is why the technique is in every GPS receiver, drone autopilot, and tracking radar. A GPS receiver measures pseudoranges, not velocity, yet reports your speed; a radar measures range and bearing, yet reports a heading. Same mechanism.

**It is not magic, and the condition has a name.** Hidden states are recoverable only when they are **observable** — when the dynamics genuinely propagate their influence into something measurable. Add a state that never affects any measurement, directly or through $F$, and the filter will report the prior back to you forever, with the covariance politely never shrinking. Observability is a checkable property of the $(F, H)$ pair, and it is the precise statement of what "for free" costs.

Notice too that the velocity estimate is visibly noisier than the position estimate and lags the sharper changes. That is expected: it is inferred at second hand, through accumulated evidence, so it takes several steps of consistent innovations to move. Anything derived rather than measured arrives late and less certain — a general property of these filters, not a defect of this one.

### 3.2. Tuning $Q$ and $R$

In practice $R$ comes from your sensor's datasheet; $Q$ is the honest confession of how much your model lies. The failure modes:

- **$Q$ too small** — the filter grows overconfident in its model and *ignores* measurements: smooth but lagging, blind to maneuvers.
- **$Q$ too large** — the filter distrusts its model and chases every noisy measurement: jittery, barely better than raw data.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three values of $Q$, one clear U-shape: RMSE **1.207** when $Q$ is far too small, **0.491** when matched, **0.842** when far too large. Mistuning in *either* direction costs accuracy, and the optimum sits where `q_accel` equals the acceleration noise the simulator actually injected — 0.6. When your model of the world matches the world, the filter is optimal; the further you drift, the more you pay.

Note the reference point: the raw measurements had RMSE 1.433. So the overconfident filter at 1.207 is barely better than not filtering at all, despite doing all the work.

**The asymmetry is the interesting part.** Being overconfident (1.207) hurts more than being jittery (0.842), and the mechanism is worth stating. With $Q$ too small the filter believes its own model almost completely, so the gain collapses toward zero and it *stops listening* to measurements — it produces a beautifully smooth trajectory that lags every maneuver, because it has decided in advance that maneuvers do not happen. With $Q$ too large it distrusts its model and chases each noisy measurement, which is ugly but at least still responsive; it degrades toward the raw data rather than toward a confident fiction.

The general lesson generalises past Kalman filters: **underestimating your own uncertainty is the more dangerous error**. A system that reports tight error bars around a wrong answer is worse than one that reports honest, wide ones, because downstream consumers act on the confidence. In a tracking system this shows up as a filter that smoothly and confidently follows a target that has already turned.

**A caveat on the middle row.** 0.491 is the *matched* case, where the filter was handed the same acceleration noise the simulator used. That is the best case and it is not available in real work: $R$ can usually be measured from a stationary sensor or read off a datasheet, but $Q$ is a modelling choice with no ground truth to look up. In practice it is tuned on validation data or estimated adaptively from the innovation sequence — if the innovations are larger than the filter's own $S = HPH^\top + R$ predicts, $Q$ is too small. Treat 0.491 as an upper bound on what tuning can achieve, not as a number you should expect to hit.

## 4. Conclusion

The Kalman filter is the predict/correct loop of the [APA workshop](./Intro_AdFilt_APA.ipynb) elevated to a *belief update*: gains derived from modeled uncertainty instead of tuned by hand — optimal when the model is linear and the noise Gaussian. When those assumptions crack, the ideas generalize (EKF/UKF for nonlinearity, and particle filters beyond that).

---
## Where next

- [Recurrent Neural Networks](./README.md#workshop-3--recurrent-neural-networks-available) — an RNN is a *learned, nonlinear* state-space model; the hidden state lives on.
- [Measure Theory → Random Variables](../Intro_Math/Analysis/README.md) — the probability foundations under $Q$, $R$, and "optimal."